# Lab: Egg Quality Classification using Color Histograms and Random Forest

**Duration:** ~30 minutes

**Objective:** Classify egg images into three mutation classes (Mutu A, B, C) using color histogram features and a Random Forest classifier.

## Pipeline
1. Load & visualize sample images
2. Segment eggs from background using thresholding
3. Extract color histograms per class
4. Split data into Train / Validation / Test
5. Train a Random Forest on histogram features
6. Evaluate: Accuracy, Precision, Recall, F1-Score, Confusion Matrix

## Step 0 — Imports and Configuration

## Step 0a — Clone Dataset from GitHub

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# ── Clone the dataset ──
!git clone https://github.com/DoreaLab/digag.git /content/digag 2>/dev/null || echo "Repository already cloned."

# ── Configuration ──
DATA_DIR = "/content/digag/Data_eggs"
CLASSES = ["Mutu A", "Mutu B", "Mutu C"]
IMG_SIZE = (256, 256)
SEED = 42
N_BINS = 64

print(f"DATA_DIR = {DATA_DIR}")
print(f"Classes: {CLASSES}")
for cls in CLASSES:
    n = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f"  {cls}: {n} images")

## Step 1 — Load and Visualize Sample Images

Let's look at a few examples from each class to understand visual differences.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))

for row, cls in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, cls)
    files = sorted(os.listdir(folder))[:5]
    for col, fname in enumerate(files):
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{cls}" if col == 0 else "")
        axes[row, col].axis("off")

fig.suptitle("Sample Images per Class", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## Step 2 — Segment Eggs from Background Using Thresholding

We convert images to grayscale and apply Otsu's thresholding to isolate the egg region from the background. This ensures our color histograms only capture egg pixels.

In [ ]:
def segment_egg(img_bgr, morph_ksize=15):
    """Segment the egg from the background using Otsu thresholding.
    Returns a binary mask (255 = egg, 0 = background)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # Gaussian blur to smooth noise
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    # Otsu threshold
    thresh_val, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Morphological closing to fill holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_ksize, morph_ksize))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask, thresh_val

# Demonstrate segmentation on one image per class
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row, cls in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, cls)
    fname = sorted(os.listdir(folder))[0]
    img = cv2.imread(os.path.join(folder, fname))
    img = cv2.resize(img, IMG_SIZE)
    mask, tval = segment_egg(img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    segmented = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
    
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f"{cls} — Original")
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"Mask (Otsu T={tval:.0f})")
    axes[row, 2].imshow(segmented)
    axes[row, 2].set_title("Segmented Egg")
    for ax in axes[row]:
        ax.axis("off")

fig.suptitle("Egg Segmentation via Otsu Thresholding", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## Step 3 — Extract Color Histograms

For each image, we:
1. Resize and segment the egg
2. Compute the RGB histogram (only on egg pixels)
3. Normalize the histogram so it sums to 1 (probability distribution)
4. Concatenate R, G, B histograms into a single feature vector

Feature vector size: `N_BINS × 3` = 192 features

In [ ]:
def extract_histogram(img_bgr, mask, n_bins=N_BINS):
    """Extract normalized RGB histogram from masked region."""
    hists = []
    for ch in range(3):  # B, G, R
        h = cv2.calcHist([img_bgr], [ch], mask, [n_bins], [0, 256])
        h = h.flatten()
        h = h / (h.sum() + 1e-8)  # normalize to distribution
        hists.append(h)
    return np.concatenate(hists)  # shape: (n_bins * 3,)

# Extract features for all images
features = []
labels = []
label_names = []

for class_idx, cls in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, cls)
    files = sorted(os.listdir(folder))
    print(f"Processing {cls} ({len(files)} images)...", end=" ")
    count = 0
    for fname in files:
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        mask, _ = segment_egg(img)
        hist = extract_histogram(img, mask)
        features.append(hist)
        labels.append(class_idx)
        count += 1
    print(f"{count} loaded.")

X = np.array(features)
y = np.array(labels)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {dict(zip(CLASSES, np.bincount(y)))}")

### Plot Average Color Histograms per Class

This shows how the color distributions differ between classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
channel_names = ["Blue", "Green", "Red"]
channel_colors = ["blue", "green", "red"]

for ch_idx in range(3):
    ax = axes[ch_idx]
    start = ch_idx * N_BINS
    end = start + N_BINS
    bins_x = np.linspace(0, 255, N_BINS)
    
    for class_idx, cls in enumerate(CLASSES):
        mask_cls = y == class_idx
        mean_hist = X[mask_cls, start:end].mean(axis=0)
        ax.plot(bins_x, mean_hist, label=cls, linewidth=2)
    
    ax.set_title(f"{channel_names[ch_idx]} Channel", fontsize=14)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Normalized Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("Average Color Histogram per Class (Segmented Egg Region)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay all 3 channels per class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins_x = np.linspace(0, 255, N_BINS)

for class_idx, cls in enumerate(CLASSES):
    ax = axes[class_idx]
    mask_cls = y == class_idx
    for ch_idx, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
        start = ch_idx * N_BINS
        end = start + N_BINS
        mean_hist = X[mask_cls, start:end].mean(axis=0)
        ax.fill_between(bins_x, mean_hist, alpha=0.3, color=ch_color)
        ax.plot(bins_x, mean_hist, color=ch_color, label=ch_name, linewidth=1.5)
    ax.set_title(cls, fontsize=14)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Normalized Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("RGB Distribution per Class", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Step 4 — Train / Validation / Test Split

We split the data into:
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%

Stratified split to maintain class balance.

In [ ]:
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Second split: 50/50 of temp → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Class distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, labels_split) in zip(axes, [("Train", y_train), ("Validation", y_val), ("Test", y_test)]):
    counts = np.bincount(labels_split, minlength=3)
    ax.bar(CLASSES, counts, color=["#4C72B0", "#55A868", "#C44E52"])
    ax.set_title(f"{name} (n={len(labels_split)})")
    ax.set_ylabel("Count")
    for i, c in enumerate(counts):
        ax.text(i, c + 1, str(c), ha="center", fontweight="bold")

fig.suptitle("Class Distribution per Split", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 5 — Train a Random Forest Classifier

The input features are the normalized color histogram distributions (192 features = 64 bins × 3 channels).

The target is the egg class (Mutu A, B, or C).

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    random_state=SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Validation performance (for tuning)
y_val_pred = rf.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=CLASSES))

### Feature Importance

Which histogram bins are most discriminative?

In [ ]:
importances = rf.feature_importances_
fig, ax = plt.subplots(figsize=(14, 4))

# Color-code by channel
colors = ["blue"] * N_BINS + ["green"] * N_BINS + ["red"] * N_BINS
ax.bar(range(len(importances)), importances, color=colors, alpha=0.7, width=1.0)
ax.set_xlabel("Feature Index (B: 0-63 | G: 64-127 | R: 128-191)")
ax.set_ylabel("Importance")
ax.set_title("Random Forest Feature Importances by Histogram Bin")
ax.axvline(x=N_BINS - 0.5, color="black", linestyle="--", alpha=0.5)
ax.axvline(x=2 * N_BINS - 0.5, color="black", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Step 6 — Test Set Evaluation

Final evaluation on the held-out test set.

In [ ]:
y_test_pred = rf.predict(X_test)

acc  = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, average="weighted")
rec  = recall_score(y_test, y_test_pred, average="weighted")
f1   = f1_score(y_test, y_test_pred, average="weighted")

print("=" * 45)
print("       TEST SET METRICS")
print("=" * 45)
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}  (weighted)")
print(f"  Recall:    {rec:.4f}  (weighted)")
print(f"  F1-Score:  {f1:.4f}  (weighted)")
print("=" * 45)

print(f"\nPer-class Report:\n")
print(classification_report(y_test, y_test_pred, target_names=CLASSES))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp1.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (Counts)")

# Normalized (recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASSES)
disp2.plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Confusion Matrix (Normalized)")

fig.suptitle("Test Set Confusion Matrix", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Step | Description |
|------|-------------|
| 1 | Loaded and visualized 601 egg images across 3 classes |
| 2 | Segmented eggs from background using Otsu thresholding |
| 3 | Extracted normalized RGB histograms (192 features) as descriptors |
| 4 | Split data: 70% Train / 15% Validation / 15% Test (stratified) |
| 5 | Trained a Random Forest (200 trees) on histogram features |
| 6 | Evaluated with Accuracy, Precision, Recall, F1, and Confusion Matrix |

### Key Takeaways
- Color histograms capture the distribution of pixel intensities — a simple but effective descriptor for egg classification.
- Segmentation ensures we only analyze egg pixels, removing background noise.
- Random Forest works well with histogram features because it handles non-linear decision boundaries across many features.
- The normalized confusion matrix reveals which classes are most commonly confused.